# Experiment 6 - End-to-End Study of RNN, LSTM and GRU for Sequence Learning and Video Understanding
**CS3807 - Deep Learning Laboratory**

Complete notebook (Sections 1-30 of the manual). Run **Runtime -> Change runtime type -> T4 GPU**, then **Run all**.

| Part | Content | Manual sections |
|---|---|---|
| 1 | UCI HAR: preprocessing, Plot 1, BPTT exercise | 3-8 |
| 2 | RNN / LSTM / GRU training, Plots 2-5, comparison | 9-16 |
| 3 | Effect of sequence length, Plot 6 | 17 |
| 4 | Video understanding (UCF101 + MobileNetV2 + LSTM/GRU), Plots 7-9 | 18-21 |
| 5 | Sequence-to-sequence reversal | 22-24 |
| 6 | Consolidated results + inference guide | 25-26 |

Every number in the tables comes from the code. Nothing is typed in by hand.

In [ ]:
import os, re, glob, time, random, zipfile, tarfile, warnings, shutil, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support, classification_report)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
set_seed()

os.makedirs("lab6_plots", exist_ok=True)
def save_fig(name):
    """Saves the current figure as EPS (for the report) and PNG (for quick viewing)."""
    plt.savefig(f"lab6_plots/{name}.eps", format="eps", dpi=600, bbox_inches="tight")
    plt.savefig(f"lab6_plots/{name}.png", dpi=150, bbox_inches="tight")

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

# PART 1 - UCI HAR dataset: loading, preprocessing, visualisation (Sections 3-8)

In [ ]:
# ---- Locate / obtain the UCI HAR dataset --------------------------------------------------
DATA_DIR = "UCI HAR Dataset"

def _download(url, out):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(out, "wb") as f:
        shutil.copyfileobj(r, f)

if not os.path.isdir(DATA_DIR):
    if not os.path.exists("UCI HAR Dataset.zip"):
        if not os.path.exists("uci_har_outer.zip"):
            print("Downloading UCI HAR dataset ...")
            _download("https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
                      "uci_har_outer.zip")
        with zipfile.ZipFile("uci_har_outer.zip") as z:
            z.extractall(".")
    if not os.path.isdir(DATA_DIR) and os.path.exists("UCI HAR Dataset.zip"):
        with zipfile.ZipFile("UCI HAR Dataset.zip") as z:
            z.extractall(".")
assert os.path.isdir(DATA_DIR), "UCI HAR Dataset folder not found - upload the dataset zip to /content and rerun."
print("Dataset folder:", DATA_DIR, "->", os.listdir(DATA_DIR))

In [ ]:
sensor_names = ["body_acc_x", "body_acc_y", "body_acc_z",
                "body_gyro_x", "body_gyro_y", "body_gyro_z",
                "total_acc_x", "total_acc_y", "total_acc_z"]

def load_signals(split):
    sig_dir = os.path.join(DATA_DIR, split, "Inertial Signals")
    chans = [np.loadtxt(os.path.join(sig_dir, f"{s}_{split}.txt")) for s in sensor_names]   # 9 x (N,128)
    return np.transpose(np.array(chans), (1, 2, 0)).astype("float32")                        # (N,128,9)

X_train_original = load_signals("train")
y_train_original = np.loadtxt(os.path.join(DATA_DIR, "train", "y_train.txt"), dtype=int)

activity_labels = {}
with open(os.path.join(DATA_DIR, "activity_labels.txt")) as f:
    for line in f:
        k, v = line.strip().split()
        activity_labels[int(k)] = v
HAR_CLASSES = [activity_labels[i] for i in range(1, 7)]
print("Raw training windows:", X_train_original.shape, "| labels:", y_train_original.shape)
print(activity_labels)

### Balanced subset (2500 windows), 70/15/15 split, normalisation with TRAINING statistics only

In [ ]:
rng = np.random.default_rng(SEED)
N_SUBSET = 2500
per_class = N_SUBSET // 6
idx = []
for c in range(1, 7):
    ci = np.where(y_train_original == c)[0]
    idx.extend(rng.choice(ci, size=min(per_class, len(ci)), replace=False))
idx = rng.permutation(np.array(idx))
X_subset, y_subset = X_train_original[idx], y_train_original[idx]

X_train, X_temp, y_train, y_temp = train_test_split(X_subset, y_subset, test_size=0.30,
                                                    random_state=SEED, stratify=y_subset)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50,
                                                random_state=SEED, stratify=y_temp)

train_mean = X_train.mean(axis=(0, 1), keepdims=True)
train_std = X_train.std(axis=(0, 1), keepdims=True)
train_std = np.where(train_std == 0, 1, train_std)
X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

y_train_m, y_val_m, y_test_m = y_train - 1, y_val - 1, y_test - 1      # labels 0..5 for the model

print("Input tensor shape:")
print("Training   :", X_train.shape)
print("Validation :", X_val.shape)
print("Testing    :", X_test.shape)
print("\nNumber of classes:", len(np.unique(y_train)))
print("Number of features per time step:", X_train.shape[2])
print("Sequence length:", X_train.shape[1])
print("\nClass distribution (train / val / test):")
print(pd.DataFrame({"train": np.bincount(y_train_m), "val": np.bincount(y_val_m),
                    "test": np.bincount(y_test_m)}, index=HAR_CLASSES))

### Plot 1 - Sensor signal vs time (3 activities x 3 channels)

In [ ]:
sel_classes = [1, 4, 6]                # WALKING, SITTING, LAYING
sel_channels = [0, 3, 6]
chan_names = ["Body Acceleration X", "Body Gyroscope X", "Total Acceleration X"]
t = np.arange(1, 129)

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
for ax, c in zip(axes, sel_classes):
    sample = X_train[np.where(y_train == c)[0][0]]
    for ch, nm in zip(sel_channels, chan_names):
        ax.plot(t, sample[:, ch], label=nm)
    ax.set_title(activity_labels[c], fontsize=14); ax.set_ylabel("Normalized value", fontsize=12)
    ax.grid(alpha=.3); ax.legend(fontsize=9, loc="upper right")
axes[-1].set_xlabel("Time step", fontsize=13)
plt.suptitle("Plot 1: Sensor signals versus time", fontsize=15)
plt.tight_layout(); save_fig("plot1_sensor_signals"); plt.show()

print("Std-dev of each channel per activity (movement intensity):")
rows = {}
for c in sel_classes:
    s = X_train[y_train == c]
    rows[activity_labels[c]] = [float(s[:, :, ch].std(axis=1).mean()) for ch in sel_channels]
print(pd.DataFrame(rows, index=chan_names).T.round(3))

### Numerical exercise (Section 8): manual RNN calculation vs program

In [ ]:
x_seq = [0.5, 0.7, 0.2]; Wx, Wh, b = 0.5, 0.8, 0.1
h = 0.0; manual = []
print("Manual recurrence  h_t = tanh(Wx*x_t + Wh*h_(t-1) + b)")
for i, xt in enumerate(x_seq, 1):
    pre = Wx * xt + Wh * h + b
    h = float(np.tanh(pre)); manual.append(h)
    print(f"  t={i}: pre-activation = {pre:.6f}  ->  h{i} = tanh({pre:.6f}) = {h:.6f}")

chk = keras.Sequential([layers.Input(shape=(3, 1)),
                        layers.SimpleRNN(1, activation="tanh", return_sequences=True)])
chk.layers[0].set_weights([np.array([[Wx]], "float32"), np.array([[Wh]], "float32"), np.array([b], "float32")])
prog = chk.predict(np.array(x_seq, "float32").reshape(1, 3, 1), verbose=0).ravel()
print("\nKeras SimpleRNN output :", np.round(prog, 6))
print("Manual calculation     :", np.round(manual, 6))
print("Match:", np.allclose(prog, manual, atol=1e-5))

# PART 2 - RNN vs LSTM vs GRU (Sections 9-16)
Identical input, output layer, optimiser (Adam, 1e-3), batch size (32), epochs (30), dropout (0.2), 32 recurrent units. **Only the recurrent layer changes.**
The same training function also produces the sequence-length study (T = 32, 64, 128) - the T = 128 runs are the main results.

In [ ]:
def build_har_model(kind, T):
    cell = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[kind]
    m = keras.Sequential([
        layers.Input(shape=(T, 9)),
        cell(32, activation="tanh"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(6, activation="softmax")], name=f"{kind}_T{T}")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def train_eval_har(kind, T, epochs=30):
    set_seed()
    m = build_har_model(kind, T)
    t0 = time.time()
    h = m.fit(X_train[:, :T], y_train_m, validation_data=(X_val[:, :T], y_val_m),
              epochs=epochs, batch_size=32, verbose=0)
    tt = time.time() - t0
    pred = m.predict(X_test[:, :T], verbose=0).argmax(1)
    return dict(model=m, hist=h.history, time=tt, pred=pred,
                acc=accuracy_score(y_test_m, pred),
                prec=precision_score(y_test_m, pred, average="macro", zero_division=0),
                rec=recall_score(y_test_m, pred, average="macro", zero_division=0),
                f1=f1_score(y_test_m, pred, average="macro", zero_division=0),
                params=m.count_params(),
                cm=confusion_matrix(y_test_m, pred, labels=range(6)))

# Warm-up (not timed) so that one-off GPU/cuDNN initialisation does not distort the first model's training time
for _k in ["RNN", "LSTM", "GRU"]:
    _m = build_har_model(_k, 128)
    _m.fit(X_train[:64], y_train_m[:64], epochs=1, batch_size=32, verbose=0)

har = {}
for T in [128, 64, 32]:
    for kind in ["RNN", "LSTM", "GRU"]:
        har[(kind, T)] = r = train_eval_har(kind, T)
        print(f"T={T:3d} | {kind:4s} | acc={r['acc']*100:6.2f}% | macro-F1={r['f1']*100:6.2f}% | "
              f"params={r['params']:,} | time={r['time']:.1f}s")

### Plots 2 and 3 - training / validation loss and accuracy (each model, T = 128)

In [ ]:
def plot_curves(kind):
    h = har[(kind, 128)]["hist"]
    plt.figure(figsize=(9, 5.5))
    plt.plot(h["loss"], label="Training loss"); plt.plot(h["val_loss"], label="Validation loss")
    plt.xlabel("Epoch", fontsize=14); plt.ylabel("Loss", fontsize=14)
    plt.title(f"Plot 2: {kind} training and validation loss", fontsize=14)
    plt.legend(fontsize=12); plt.grid(alpha=.3); plt.tight_layout()
    save_fig(f"plot2_{kind.lower()}_loss"); plt.show()

    plt.figure(figsize=(9, 5.5))
    plt.plot(np.array(h["accuracy"]) * 100, label="Training accuracy")
    plt.plot(np.array(h["val_accuracy"]) * 100, label="Validation accuracy")
    plt.xlabel("Epoch", fontsize=14); plt.ylabel("Accuracy (%)", fontsize=14)
    plt.title(f"Plot 3: {kind} training and validation accuracy", fontsize=14)
    plt.legend(fontsize=12); plt.grid(alpha=.3); plt.tight_layout()
    save_fig(f"plot3_{kind.lower()}_accuracy"); plt.show()

for kind in ["RNN", "LSTM", "GRU"]:
    plot_curves(kind)

In [ ]:
def convergence_report(kind):
    h = har[(kind, 128)]["hist"]
    tl, vl = np.array(h["loss"]), np.array(h["val_loss"])
    ta, va = np.array(h["accuracy"]) * 100, np.array(h["val_accuracy"]) * 100
    best_ep = int(vl.argmin()) + 1
    conv_ep = int(np.argmax(va >= 0.95 * va.max())) + 1
    gap = ta[-1] - va[-1]
    if ta[-1] < 70 and va[-1] < 70:
        verdict = "UNDERFITTING indicator (accuracy low on both training and validation data)"
    elif gap > 10 or (vl[-1] > 1.15 * vl.min() and best_ep < len(vl) - 3):
        verdict = "OVERFITTING indicator (large generalisation gap or validation loss rising after its minimum)"
    else:
        verdict = "no strong over/underfitting (small generalisation gap)"
    print(f"--- {kind} ---")
    print(f"  final train loss/acc : {tl[-1]:.4f} / {ta[-1]:.2f}%   | final val loss/acc : {vl[-1]:.4f} / {va[-1]:.2f}%")
    print(f"  lowest val loss      : {vl.min():.4f} at epoch {best_ep}")
    print(f"  val acc >= 95% of its maximum first reached at epoch {conv_ep}")
    print(f"  generalisation gap (train acc - val acc) : {gap:.2f} percentage points")
    print(f"  epoch-to-epoch val-loss fluctuation (std of last 10 epochs) : {vl[-10:].std():.4f}")
    print(f"  verdict (heuristic) : {verdict}\n")

for kind in ["RNN", "LSTM", "GRU"]:
    convergence_report(kind)

### Plot 4 - Confusion matrices + automatic error analysis

In [ ]:
def plot_cm(cm, names, title, fname, figsize=(9, 7.5)):
    plt.figure(figsize=figsize)
    plt.imshow(cm, cmap="Blues"); plt.colorbar()
    plt.xticks(range(len(names)), names, rotation=45, ha="right", fontsize=11)
    plt.yticks(range(len(names)), names, fontsize=11)
    plt.xlabel("Predicted class", fontsize=13); plt.ylabel("True class", fontsize=13)
    plt.title(title, fontsize=14)
    thr = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=11,
                     color="white" if cm[i, j] > thr else "black")
    plt.tight_layout(); save_fig(fname); plt.show()

def cm_report(cm, names, tag=""):
    recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
    errors = cm.sum(1) - cm.diagonal()
    off = cm.copy(); np.fill_diagonal(off, 0)
    print(f"[{tag}] Highest recognition rate : {names[int(recall.argmax())]} ({recall.max()*100:.1f}%)")
    print(f"[{tag}] Lowest recognition rate  : {names[int(recall.argmin())]} ({recall.min()*100:.1f}%)")
    print(f"[{tag}] Most misclassifications  : {names[int(errors.argmax())]} ({errors.max()} errors)")
    if off.max() > 0:
        i, j = np.unravel_index(off.argmax(), off.shape)
        print(f"[{tag}] Most frequent confusion  : {names[i]} -> {names[j]} ({off[i, j]} times)")
    else:
        print(f"[{tag}] No misclassification.")
    return names[int(errors.argmax())]

worst = {}
for kind in ["RNN", "LSTM", "GRU"]:
    plot_cm(har[(kind, 128)]["cm"], HAR_CLASSES, f"Plot 4: {kind} confusion matrix", f"plot4_{kind.lower()}_confusion_matrix")
    worst[kind] = cm_report(har[(kind, 128)]["cm"], HAR_CLASSES, kind)
    print()
print("Is the class with most errors the same for all three models?", len(set(worst.values())) == 1, worst)

### Performance table, RNN vs LSTM vs GRU table (Section 16), Plot 5

In [ ]:
kinds = ["RNN", "LSTM", "GRU"]
perf = pd.DataFrame({
    "Metric": ["Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)", "Parameters", "Training Time (s)"],
    **{k: [har[(k, 128)]["acc"] * 100, har[(k, 128)]["prec"] * 100, har[(k, 128)]["rec"] * 100,
           har[(k, 128)]["f1"] * 100, har[(k, 128)]["params"], har[(k, 128)]["time"]] for k in kinds}
}).set_index("Metric")
print("Section 14 - performance on the independent test set"); display(perf.round(2))

def _r(k, key, fmt):
    return fmt.format(har[(k, 128)][key] * (100 if key == "f1" else 1))

prop = pd.DataFrame({
    "Property": ["Hidden state", "Cell state", "Forget gate", "Input gate", "Output gate", "Update gate", "Reset gate",
                 "Parameters", "Training time (s)", "Test F1-score (%)"],
    "RNN":  ["Yes", "No", "No", "No", "No", "No", "No", _r("RNN", "params", "{:,}"), _r("RNN", "time", "{:.1f}"), _r("RNN", "f1", "{:.2f}")],
    "LSTM": ["Yes", "Yes", "Yes", "Yes", "Yes", "No", "No", _r("LSTM", "params", "{:,}"), _r("LSTM", "time", "{:.1f}"), _r("LSTM", "f1", "{:.2f}")],
    "GRU":  ["Yes", "No", "No", "No", "No", "Yes", "Yes", _r("GRU", "params", "{:,}"), _r("GRU", "time", "{:.1f}"), _r("GRU", "f1", "{:.2f}")],
}).set_index("Property")
print("\nSection 16 - RNN vs LSTM vs GRU"); display(prop)

# Plot 5: accuracy / F1 and normalised computational cost
x = np.arange(3); w = 0.35
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].bar(x - w/2, [har[(k, 128)]["acc"] * 100 for k in kinds], w, label="Test accuracy")
ax[0].bar(x + w/2, [har[(k, 128)]["f1"] * 100 for k in kinds], w, label="Macro F1")
ax[0].set_xticks(x); ax[0].set_xticklabels(kinds, fontsize=12)
ax[0].set_ylabel("Score (%)", fontsize=13); ax[0].set_title("Predictive performance", fontsize=14)
ax[0].legend(fontsize=11); ax[0].grid(axis="y", alpha=.3)
pn = np.array([har[(k, 128)]["params"] for k in kinds], float); tn = np.array([har[(k, 128)]["time"] for k in kinds], float)
ax[1].bar(x - w/2, pn / pn.max(), w, label="Parameters (normalised)")
ax[1].bar(x + w/2, tn / tn.max(), w, label="Training time (normalised)")
ax[1].set_xticks(x); ax[1].set_xticklabels(kinds, fontsize=12)
ax[1].set_ylabel("Relative to maximum", fontsize=13); ax[1].set_title("Computational cost", fontsize=14)
ax[1].legend(fontsize=11); ax[1].grid(axis="y", alpha=.3)
plt.suptitle("Plot 5: Model performance comparison", fontsize=15)
plt.tight_layout(); save_fig("plot5_model_comparison"); plt.show()

# PART 3 - Effect of sequence length (Section 17)
Sequences are truncated to the first T time steps (same windows, same labels, same training protocol).

In [ ]:
seq_table = pd.DataFrame({
    "Sequence Length": [32, 64, 128],
    "RNN F1 (%)":  [har[("RNN", T)]["f1"] * 100 for T in [32, 64, 128]],
    "LSTM F1 (%)": [har[("LSTM", T)]["f1"] * 100 for T in [32, 64, 128]],
    "GRU F1 (%)":  [har[("GRU", T)]["f1"] * 100 for T in [32, 64, 128]],
}).set_index("Sequence Length")
display(seq_table.round(2))

cost = pd.DataFrame({"Sequence Length": [32, 64, 128],
                     **{f"{k} time (s)": [har[(k, T)]["time"] for T in [32, 64, 128]] for k in kinds}}).set_index("Sequence Length")
print("Training time versus sequence length (computational cost):"); display(cost.round(1))

plt.figure(figsize=(9, 5.5))
for k in kinds:
    plt.plot([32, 64, 128], [har[(k, T)]["f1"] * 100 for T in [32, 64, 128]], marker="o", label=k)
plt.xticks([32, 64, 128]); plt.xlabel("Sequence length T", fontsize=14); plt.ylabel("Test macro F1 (%)", fontsize=14)
plt.title("Plot 6: Sequence length vs test F1-score", fontsize=14)
plt.legend(fontsize=12); plt.grid(alpha=.3); plt.tight_layout(); save_fig("plot6_sequence_length_vs_f1"); plt.show()

# PART 4 - Video Understanding: UCF101 -> 10 frames -> frozen MobileNetV2 -> LSTM/GRU (Sections 18-21)
### Dataset: small UCF101 subset (Hugging Face mirror, about 171 MB, already split into train/val/test without leakage between clips of the same scene)

In [ ]:
from huggingface_hub import hf_hub_download

EXTRACT_DIR = "/content/ucf_hf"
def find_train_dirs():
    return sorted(d for d in glob.glob(f"{EXTRACT_DIR}/**/train", recursive=True) if os.path.isdir(d))

if not find_train_dirs():
    tar_path = hf_hub_download(repo_id="sayakpaul/ucf101-subset", filename="UCF101_subset.tar.gz", repo_type="dataset")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall(EXTRACT_DIR)

VIDEO_ROOT = os.path.dirname(find_train_dirs()[0])
available = sorted(os.listdir(os.path.join(VIDEO_ROOT, "train")))
print("Dataset root:", VIDEO_ROOT)
print("Classes available in this subset:", available)

preferred = ["Basketball", "BasketballDunk", "Archery", "BabyCrawling", "BandMarching", "BenchPress"]
VIDEO_CLASSES = [c for c in preferred if c in available][:5]
VIDEO_CLASSES += [c for c in available if c not in VIDEO_CLASSES][:5 - len(VIDEO_CLASSES)]
NUM_CLASSES_VIDEO = len(VIDEO_CLASSES)
print("Selected classes:", VIDEO_CLASSES)

video_list = []      # (path, class, split)
for split in ["train", "val", "test"]:
    for cls in VIDEO_CLASSES:
        for p in sorted(glob.glob(f"{VIDEO_ROOT}/{split}/{cls}/*")):
            if p.lower().endswith((".avi", ".mp4", ".mov")):
                video_list.append((p, cls, split))
assert len(video_list) > 0, "No videos found"
print("Total videos:", len(video_list))
print(pd.crosstab([v[1] for v in video_list], [v[2] for v in video_list]))

### Frame sampling (10 uniform frames, 224x224x3) and frozen MobileNetV2 feature extraction

In [ ]:
NUM_FRAMES, IMG = 10, 224

def sample_video_frames(path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(n - 1, 0), num_frames).astype(int)
    frames, last = [], None
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if ok:
            f = cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (IMG, IMG)); last = f
        elif last is not None:
            f = last
        else:
            f = np.zeros((IMG, IMG, 3), np.uint8)
        frames.append(f)
    cap.release()
    return np.stack(frames).astype(np.uint8)                      # (10, 224, 224, 3)

cnn_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(IMG, IMG, 3))
cnn_model.trainable = False                                        # frozen: used only as a feature extractor
CNN_FEATURE_DIM = cnn_model.output_shape[-1]
print("CNN: MobileNetV2 | trainable:", cnn_model.trainable, "| feature dimension D =", CNN_FEATURE_DIM)

CACHE = "/content/video_features_" + "_".join(VIDEO_CLASSES) + ".npz"
if os.path.exists(CACHE):
    d = np.load(CACHE, allow_pickle=True)
    v_feats, v_labels, v_splits = d["feats"], d["labels"], d["splits"]
    vis_frames, vis_name = d["vis_frames"], str(d["vis_name"])
    feat_time = float(d["feat_time"]); print("Loaded cached features.")
else:
    t0 = time.time()
    v_feats, v_labels, v_splits, vis_frames, vis_name = [], [], [], None, None
    for path, cls, split in tqdm(video_list, desc="Extracting CNN features"):
        fr = sample_video_frames(path)
        if vis_frames is None:
            vis_frames, vis_name = fr.copy(), f"{cls}: {os.path.basename(path)}"
        v_feats.append(cnn_model.predict(preprocess_input(fr.astype("float32")), verbose=0))   # (10, D)
        v_labels.append(cls); v_splits.append(split)
    v_feats = np.array(v_feats, dtype=np.float32)
    v_labels, v_splits = np.array(v_labels), np.array(v_splits)
    feat_time = time.time() - t0
    np.savez(CACHE, feats=v_feats, labels=v_labels, splits=v_splits, vis_frames=vis_frames,
             vis_name=vis_name, feat_time=feat_time)

vy = np.array([VIDEO_CLASSES.index(c) for c in v_labels])
tr, va, te = v_splits == "train", v_splits == "val", v_splits == "test"
assert tr.sum() > 0 and va.sum() > 0 and te.sum() > 0, "An empty split was found"
Xv_train, yv_train = v_feats[tr], vy[tr]
Xv_val, yv_val = v_feats[va], vy[va]
Xv_test, yv_test = v_feats[te], vy[te]

vm = Xv_train.mean(axis=(0, 1), keepdims=True); vs = Xv_train.std(axis=(0, 1), keepdims=True) + 1e-6
Xv_train, Xv_val, Xv_test = [(a - vm) / vs for a in (Xv_train, Xv_val, Xv_test)]

print("\n===== REQUIRED OBSERVATION =====")
print(f"Frame                      : {IMG} x {IMG} x 3")
print(f"CNN feature dimension D    : {CNN_FEATURE_DIM}")
print(f"Input to recurrent network : (10, {CNN_FEATURE_DIM})  ->  batch: (B, 10, {CNN_FEATURE_DIM})")
print("Train :", Xv_train.shape, "| Val :", Xv_val.shape, "| Test :", Xv_test.shape)
print(f"Feature-extraction time    : {feat_time:.1f} s")

### Plot 7 - Video sample frames

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(vis_frames[i]); ax.set_title(f"Frame {i + 1}", fontsize=13); ax.axis("off")
plt.suptitle(f"Plot 7: 10 uniformly sampled frames\n{vis_name}", fontsize=15)
plt.tight_layout(); save_fig("plot7_video_sample_frames"); plt.show()

### CNN-LSTM and CNN-GRU (32 recurrent units) - trained on the frozen CNN features

In [ ]:
def build_video_model(kind):
    inp = keras.Input(shape=(NUM_FRAMES, CNN_FEATURE_DIM))
    x = layers.LSTM(32)(inp) if kind == "LSTM" else layers.GRU(32)(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES_VIDEO, activation="softmax")(x)
    m = keras.Model(inp, out, name=f"CNN_{kind}")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

video = {}
for kind in ["LSTM", "GRU"]:
    set_seed()
    m = build_video_model(kind)
    t0 = time.time()
    h = m.fit(Xv_train, yv_train, validation_data=(Xv_val, yv_val), epochs=40, batch_size=16, verbose=0)
    tt = time.time() - t0
    probs = m.predict(Xv_test, verbose=0); pred = probs.argmax(1)
    video[kind] = dict(model=m, hist=h.history, time=tt, probs=probs, pred=pred,
                       acc=accuracy_score(yv_test, pred),
                       prec=precision_score(yv_test, pred, average="macro", zero_division=0),
                       rec=recall_score(yv_test, pred, average="macro", zero_division=0),
                       f1=f1_score(yv_test, pred, average="macro", zero_division=0),
                       params=m.count_params(),
                       cm=confusion_matrix(yv_test, pred, labels=range(NUM_CLASSES_VIDEO)))
    print(f"CNN-{kind}: params={video[kind]['params']:,} | time={tt:.1f}s | "
          f"final val acc={h.history['val_accuracy'][-1]*100:.1f}% | test acc={video[kind]['acc']*100:.1f}%")

# model selection uses VALIDATION accuracy only (the test set is never used for selection)
BEST_VIDEO = max(video, key=lambda k: video[k]["hist"]["val_accuracy"][-1])
print("\nSelected recurrent architecture (best final validation accuracy): CNN-" + BEST_VIDEO)

### Plot 8 - Video training / validation curves

In [ ]:
for kind in ["LSTM", "GRU"]:
    h = video[kind]["hist"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(h["loss"], label="Training loss"); ax[0].plot(h["val_loss"], label="Validation loss")
    ax[0].set_xlabel("Epoch", fontsize=13); ax[0].set_ylabel("Loss", fontsize=13); ax[0].legend(fontsize=11); ax[0].grid(alpha=.3)
    ax[1].plot(np.array(h["accuracy"]) * 100, label="Training accuracy")
    ax[1].plot(np.array(h["val_accuracy"]) * 100, label="Validation accuracy")
    ax[1].set_xlabel("Epoch", fontsize=13); ax[1].set_ylabel("Accuracy (%)", fontsize=13); ax[1].legend(fontsize=11); ax[1].grid(alpha=.3)
    plt.suptitle(f"Plot 8: CNN-{kind} training and validation curves" + ("  (selected)" if kind == BEST_VIDEO else ""), fontsize=15)
    plt.tight_layout(); save_fig(f"plot8_video_{kind.lower()}_curves"); plt.show()

### Test evaluation, confidence, Plot 9 - Video confusion matrix

In [ ]:
vb = video[BEST_VIDEO]
print(f"===== CNN-{BEST_VIDEO} (selected) - test results =====")
print(f"Accuracy {vb['acc']*100:.2f}% | Macro precision {vb['prec']*100:.2f}% | Macro recall {vb['rec']*100:.2f}% | "
      f"Macro F1 {vb['f1']*100:.2f}% | Parameters {vb['params']:,} | Training time {vb['time']:.1f}s")
print(classification_report(yv_test, vb["pred"], labels=range(NUM_CLASSES_VIDEO),
                            target_names=VIDEO_CLASSES, zero_division=0))

plot_cm(vb["cm"], VIDEO_CLASSES, f"Plot 9: CNN-{BEST_VIDEO} video confusion matrix", "plot9_video_confusion_matrix", figsize=(8, 7))
cm_report(vb["cm"], VIDEO_CLASSES, f"CNN-{BEST_VIDEO}")

print("\n--- Sample predictions (confidence is the model's softmax output) ---")
for i in np.random.RandomState(0).choice(len(yv_test), size=min(5, len(yv_test)), replace=False):
    print(f"Predicted class : {VIDEO_CLASSES[vb['pred'][i]]}")
    print(f"Actual class    : {VIDEO_CLASSES[yv_test[i]]}")
    print(f"Confidence      : {vb['probs'][i, vb['pred'][i]]:.2f}\n")

# PART 5 - Sequence-to-sequence learning: sequence reversal (Sections 22-24)
Encoder LSTM -> context state (h, c) -> decoder LSTM, trained with **teacher forcing** (the decoder receives the true previous token during training) and evaluated with **greedy autoregressive decoding** (the decoder receives its own previous prediction).

In [ ]:
set_seed()
SEQ_LEN, NUM_DIGITS, VOCAB = 8, 9, 10        # digits 1..9, token 0 = START
N_SEQ = 10000
ids = np.random.permutation(NUM_DIGITS ** SEQ_LEN)[:N_SEQ]            # unique sequences (no duplicates across splits)
Xs = np.stack([(ids // NUM_DIGITS ** j) % NUM_DIGITS + 1 for j in range(SEQ_LEN)], axis=1).astype("int64")
Ys = Xs[:, ::-1].copy()
Dec_in = np.concatenate([np.zeros((N_SEQ, 1), "int64"), Ys[:, :-1]], axis=1)      # START + shifted target

n_tr, n_va = int(0.8 * N_SEQ), int(0.1 * N_SEQ)
s_tr, s_va, s_te = slice(0, n_tr), slice(n_tr, n_tr + n_va), slice(n_tr + n_va, N_SEQ)
print("Train / Val / Test sequences:", n_tr, n_va, N_SEQ - n_tr - n_va)
print("Example:", list(Xs[0]), "->", list(Ys[0]), "| decoder input (teacher forcing):", list(Dec_in[0]))

UNITS, EMB = 32, 16      # deliberately small so that token vs sequence accuracy can differ
enc_in = keras.Input(shape=(SEQ_LEN,), dtype="int64", name="encoder_input")
e = layers.Embedding(VOCAB, EMB)(enc_in)
_, state_h, state_c = layers.LSTM(UNITS, return_state=True, name="encoder_lstm")(e)
dec_in = keras.Input(shape=(SEQ_LEN,), dtype="int64", name="decoder_input")
d = layers.Embedding(VOCAB, EMB)(dec_in)
d = layers.LSTM(UNITS, return_sequences=True, name="decoder_lstm")(d, initial_state=[state_h, state_c])
dec_out = layers.Dense(VOCAB, activation="softmax")(d)
seq2seq = keras.Model([enc_in, dec_in], dec_out, name="seq2seq_reverse")
seq2seq.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
seq2seq.summary()

t0 = time.time()
s2s_hist = seq2seq.fit([Xs[s_tr], Dec_in[s_tr]], Ys[s_tr],
                       validation_data=([Xs[s_va], Dec_in[s_va]], Ys[s_va]),
                       epochs=25, batch_size=128, verbose=0,
                       callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)])
s2s_time = time.time() - t0
print(f"Trained {len(s2s_hist.history['loss'])} epochs in {s2s_time:.1f}s")

In [ ]:
def greedy_decode(model, X):
    dec = np.zeros((len(X), SEQ_LEN), dtype="int64")
    pred = np.zeros((len(X), SEQ_LEN), dtype="int64")
    for t in range(SEQ_LEN):
        p = model.predict([X, dec], verbose=0)
        pred[:, t] = p[:, t, :].argmax(-1)
        if t + 1 < SEQ_LEN:
            dec[:, t + 1] = pred[:, t]
    return pred

X_te_s, Y_te_s = Xs[s_te], Ys[s_te]
Y_pred_s = greedy_decode(seq2seq, X_te_s)
token_acc = float((Y_pred_s == Y_te_s).mean())
seq_acc = float((Y_pred_s == Y_te_s).all(axis=1).mean())
hh = s2s_hist.history

print(f"Token accuracy    : {token_acc*100:.2f}%")
print(f"Sequence accuracy : {seq_acc*100:.2f}%")
print(f"Training loss (final) : {hh['loss'][-1]:.4f} | Validation loss (final) : {hh['val_loss'][-1]:.4f}")
print(f"Reference: if token errors were independent, sequence accuracy would be about token_acc^L = {token_acc**SEQ_LEN*100:.2f}%")

rows = [{"Sample": i + 1, "Input Sequence": list(map(int, X_te_s[i])),
         "Expected": list(map(int, Y_te_s[i])), "Predicted Output": list(map(int, Y_pred_s[i])),
         "Correct": bool((Y_pred_s[i] == Y_te_s[i]).all())} for i in range(5)]
print("\nFive test examples:"); display(pd.DataFrame(rows).set_index("Sample"))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(hh["loss"], label="Training loss"); ax[0].plot(hh["val_loss"], label="Validation loss")
ax[0].set_xlabel("Epoch", fontsize=13); ax[0].set_ylabel("Loss", fontsize=13); ax[0].legend(fontsize=11); ax[0].grid(alpha=.3)
ax[0].set_title("Seq2seq loss", fontsize=14)
pos = (Y_pred_s == Y_te_s).mean(axis=0) * 100
ax[1].bar(range(1, SEQ_LEN + 1), pos); ax[1].set_ylim(0, 105)
ax[1].set_xlabel("Output position", fontsize=13); ax[1].set_ylabel("Token accuracy (%)", fontsize=13)
ax[1].set_title("Accuracy by output position", fontsize=14); ax[1].grid(axis="y", alpha=.3)
plt.tight_layout(); save_fig("plot_seq2seq_loss_and_position_accuracy"); plt.show()

# PART 6 - Consolidated results (Section 25)

In [ ]:
cons = pd.DataFrame([
    {"Model": "RNN", "Accuracy (%)": har[("RNN", 128)]["acc"] * 100, "Precision (%)": har[("RNN", 128)]["prec"] * 100,
     "Recall (%)": har[("RNN", 128)]["rec"] * 100, "F1 (%)": har[("RNN", 128)]["f1"] * 100, "Parameters": har[("RNN", 128)]["params"]},
    {"Model": "LSTM", "Accuracy (%)": har[("LSTM", 128)]["acc"] * 100, "Precision (%)": har[("LSTM", 128)]["prec"] * 100,
     "Recall (%)": har[("LSTM", 128)]["rec"] * 100, "F1 (%)": har[("LSTM", 128)]["f1"] * 100, "Parameters": har[("LSTM", 128)]["params"]},
    {"Model": "GRU", "Accuracy (%)": har[("GRU", 128)]["acc"] * 100, "Precision (%)": har[("GRU", 128)]["prec"] * 100,
     "Recall (%)": har[("GRU", 128)]["rec"] * 100, "F1 (%)": har[("GRU", 128)]["f1"] * 100, "Parameters": har[("GRU", 128)]["params"]},
    {"Model": f"CNN-{BEST_VIDEO} (video)", "Accuracy (%)": vb["acc"] * 100, "Precision (%)": vb["prec"] * 100,
     "Recall (%)": vb["rec"] * 100, "F1 (%)": vb["f1"] * 100, "Parameters": vb["params"]},
]).set_index("Model")
print("Consolidated classification results (test set, macro-averaged). HAR rows: 6 activities; video row: %d actions." % NUM_CLASSES_VIDEO)
display(cons.round(2))

vid_both = pd.DataFrame({f"CNN-{k}": {"Accuracy (%)": video[k]["acc"] * 100, "Macro F1 (%)": video[k]["f1"] * 100,
                                     "Parameters": video[k]["params"], "Training time (s)": video[k]["time"]} for k in video})
print("\nCNN-LSTM vs CNN-GRU on identical CNN features:"); display(vid_both.round(2))

s2s_table = pd.DataFrame({
    "Model": ["Encoder-Decoder LSTM"], "Token Accuracy (%)": [token_acc * 100], "Sequence Accuracy (%)": [seq_acc * 100],
    "Training Loss": [hh["loss"][-1]], "Validation Loss": [hh["val_loss"][-1]],
    "Parameters": [seq2seq.count_params()], "Training Time (s)": [s2s_time]}).set_index("Model")
print("\nSequence-to-sequence results:"); display(s2s_table.round(4))

In [ ]:
# ---- Save everything into one zip for the report -------------------------------------------
cons.round(4).to_csv("lab6_plots/table_consolidated.csv")
perf.round(4).to_csv("lab6_plots/table_har_performance.csv")
seq_table.round(4).to_csv("lab6_plots/table_sequence_length.csv")
s2s_table.round(4).to_csv("lab6_plots/table_seq2seq.csv")
shutil.make_archive("/content/lab6_outputs", "zip", "lab6_plots")
print("Saved:", sorted(os.listdir("lab6_plots")))
try:
    from google.colab import files
    files.download("/content/lab6_outputs.zip")
except Exception as ex:
    print("Automatic download skipped:", ex, "- use the Files panel to download /content/lab6_outputs.zip")

# Inference guide (Section 26) - write these in your own words using YOUR printed numbers/plots
For each plot cover: **(1) what it shows, (2) trend, (3) why, (4) conclusion.**

| Mandatory inference | Where to look | Key idea |
|---|---|---|
| Temporal pattern in sensor signals | Plot 1 + std table | Dynamic activities (walking, stairs) show periodic, high-variance signals; static ones (sitting, standing, laying) are nearly flat and differ mainly in the gravity component of total acceleration. Order matters because the pattern is in the *sequence* of values (periodicity, phase), not in individual samples. |
| Convergence of RNN / LSTM / GRU | Plots 2-3 + `convergence_report` | Compare the epoch at which validation loss is lowest, how smooth the curves are, and the final loss. Vanilla RNN uses one tanh transformation, so gradients shrink/explode over 128 steps (BPTT), giving slower or noisier convergence; gated models keep a more direct gradient path. |
| Over/underfitting | `convergence_report`: gap and verdict | A large train-validation gap or rising validation loss = overfitting; low accuracy on both = underfitting. |
| Activity-wise errors | Plot 4 + `cm_report` | Use YOUR matrix: which class is lowest, which pair is confused. Static postures with similar signals (sitting vs standing) and the walking family are the usual candidates - but state only what your matrix shows, and whether it is consistent across the 3 models. |
| Effect of sequence length | Plot 6 + tables | More time steps give more temporal context (whole gait cycles) but cost more training time (compare the time table). Report whether each model benefits, and whether RNN degrades more at long T. |
| Parameter / compute differences | Section 16 table, Plot 5 | With 32 units and 9 inputs: RNN has 1 transformation, GRU 3, LSTM 4, so parameters scale roughly 1 : 3 : 4 and time grows accordingly. Discuss the trade-off Performance <-> Complexity <-> Training cost. |
| LSTM cell state and gates | Section 10 | The cell state is an additive memory path (C_t = f_t*C_(t-1) + i_t*C~_t); the forget gate discards, the input gate writes, the output gate exposes - this preserves gradients over long spans. |
| Role of the CNN in video | Plot 7, Part 4 | Frozen MobileNetV2 converts each frame into a D = 1280 vector encoding spatial content (objects, scene, pose, texture); it captures no motion. |
| Role of the recurrent model in video | Plots 8-9 | The LSTM/GRU reads the 10 frame vectors in order and models how the scene evolves (temporal information), producing one clip-level representation for the softmax. |
| Token vs sequence accuracy | Part 5 output | A sequence is correct only if ALL tokens are correct, so sequence accuracy <= token accuracy and falls quickly as length grows (about p^L if errors were independent - compare with the printed reference). Autoregressive decoding also compounds errors, since a wrong token is fed to the next step. |